# gVCF to VDS

In [1]:
%%configure -f
{
    "driverMemory": "45G"
}

In [2]:
# Import and initiate HAIL
import hail as hl
hl.init(sc)

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1755333991027_0001,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

pip-installed Hail requires additional configuration options in Spark referring
  to the path to the Hail Python module directory HAIL_DIR,
  e.g. /path/to/python/site-packages/hail:
    spark.jars=HAIL_DIR/backend/hail-all-spark.jar
    spark.driver.extraClassPath=HAIL_DIR/backend/hail-all-spark.jar
    spark.executor.extraClassPath=./hail-all-spark.jarRunning on Apache Spark version 3.5.2-amzn-1
SparkUI available at http://ip-192-168-100-168.ap-southeast-1.compute.internal:35699
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.134-952ae203dbbe
LOGGING: writing to /mnt/yarn/usercache/livy/appcache/application_1755333991027_0001/container_1755333991027_0001_01_000001/hail-20250816-0859-0.2.134-952ae203dbbe.log

## Load 1KG gVCF into VDS

- List the single sample hard-filtered.gvcf.gz generated for 1000 genomes dragen 3.7.6 analysis
- Upload the list (csv file) into S3
- Set gvcf_list_path

In [3]:
# gvcf_list_path='s3://npm-grids/hebrardms/batch/sg10k_reprocess_gvcf_manifest.csv'
# vds_prefix = 's3://precise-scratch/hebrardms/SG10K_Health/VDS'
gvcf_list_path='s3://npm-grids/hebrardms/batch/sg10k_reprocess_gvcf_manifest.csv'
vds_prefix = 's3://precise-scratch/goypav/SG10K_Health/VDS'

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
# Import csv samples to ht
ht_gvcf = hl.import_table(gvcf_list_path, delimiter=',', quote = '"', no_header=True)
# Rename the columns
ht_gvcf = ht_gvcf.rename({'f0': 'bucket', 'f1': 'prefix'})
# Build S3 path
ht_gvcf = ht_gvcf.annotate(
    s3_path = hl.str('s3a://') + hl.str(ht_gvcf.bucket) + '/' + hl.str(ht_gvcf.prefix)
)

ht_gvcf.count()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

10322
2025-08-16 09:00:03.053 Hail: INFO: Reading table without type imputation
  Loading field 'f0' as type str (not specified)
  Loading field 'f1' as type str (not specified)

In [5]:
# List of S3 path
ls_gvcf = ht_gvcf.s3_path.collect()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [6]:
## Test
###
gvcf_paths=ls_gvcf[0:10] # variant_data: 13905139 rows and 10 columns in 2586 partitions ~ 30Gb ~ 2h on 9 CPU onDemand
# gvcf_paths=ls_gvcf[0:100] # variant_data: 37755085 rows and 100 columns in 2586 partitions ~ 30Gb ~ 30min on 500 CPU onDemand
gvcf_paths

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

['s3a://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB6375/6de5907f-6049-4852-99c7-adfda64d6889/output/try-1/WHB6375.hard-filtered.gvcf.gz', 's3a://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB6377/64a24a8b-25fd-42a8-93f2-22ebfd581706/output/try-1/WHB6377.hard-filtered.gvcf.gz', 's3a://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB6378/e272cfc6-81ea-49b4-8694-4413c7907e9f/output/try-1/WHB6378.hard-filtered.gvcf.gz', 's3a://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB6380/c5169919-0140-411e-bf2b-765553f35f1f/output/try-1/WHB6380.hard-filtered.gvcf.gz', 's3a://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB6381/f2a39ab6-0342-4980-8b31-791a41f4e549/output/try-1/WHB6381.hard-filtered.gvcf.gz', 's3a://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB6300/fdd1b13b-7452-45c0-954a-80560830e731/output/try-1/WHB6300.hard-filtered.gvcf.gz', 's3a://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB6385/2af836d8-43b0-442f-ba4d-6f463f24c74e/output/try-1/WHB6385.hard-filtered.gvcf.gz', 's3a:

# gvcf to VDS (new)

In [7]:
# Step 5: gVCF combiner batch 5
###

# Change the number of tasks hail is launching in parallel
hl._set_flags(spark_max_stage_parallelism='1000')

# Combine next batch of gVCFs
combiner = hl.vds.new_combiner(
    output_path=f'{vds_prefix}/SG10K_Health_gvcf1000-bf50-tr500k-sp1k_batch5.vds',
    temp_path=f'{vds_prefix}/checkpoints/',
    gvcf_paths=ls_gvcf[4000:5000],
    use_genome_default_intervals=True,
    reference_genome='GRCh38',
    branch_factor=50,
    target_records=500_000
)

combiner.run()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

2025-08-16 09:05:54.843 Hail: WARN: expected input file 's3a://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHH1773/7854cd52-7439-463e-a1d7-e3b41267527a/output/WHH1773.hard-filtered.gvcf.gz' to end in .vcf[.bgz, .gz]
2025-08-16 09:05:55.212 Hail: WARN: expected input file 's3a://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHH1773/7854cd52-7439-463e-a1d7-e3b41267527a/output/WHH1773.hard-filtered.gvcf.gz' to end in .vcf[.bgz, .gz]
2025-08-16 09:05:55.987 Hail: INFO: scanning VCF for sortedness...
2025-08-16 09:07:21.802 Hail: INFO: Coerced sorted VCF - no additional import work to do
2025-08-16 09:07:26.815 Hail: WARN: generated combiner save path of s3://precise-scratch/goypav/SG10K_Health/VDS/checkpoints/combiner-plans/vds-combiner-plan_4679873529d6462d8fcd8de5aef0ab6672c8fcf85316e1fb74ea47077c245400_0.2.134.json
2025-08-16 09:07:26.855 Hail: INFO: Running VDS combiner:
    VDS arguments: 0 datasets with 0 samples
    GVCF arguments: 1000 inputs/samples
    Branch factor: 50
    GVCF 

In [8]:
# Step 6: gVCF combiner batch 6
###

# Change the number of tasks hail is launching in parallel
hl._set_flags(spark_max_stage_parallelism='1000')

# Combine next batch of gVCFs
combiner = hl.vds.new_combiner(
    output_path=f'{vds_prefix}/SG10K_Health_gvcf1000-bf50-tr500k-sp1k_batch6.vds',
    temp_path=f'{vds_prefix}/checkpoints/',
    gvcf_paths=ls_gvcf[5000:6000],
    use_genome_default_intervals=True,
    reference_genome='GRCh38',
    branch_factor=50,
    target_records=500_000
)

combiner.run()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

2025-08-16 22:53:27.665 Hail: WARN: expected input file 's3a://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHH4954/ce17de38-54f8-45bf-82ba-ea1aec55061e/output/try-1/WHH4954.hard-filtered.gvcf.gz' to end in .vcf[.bgz, .gz]
2025-08-16 22:53:27.836 Hail: WARN: expected input file 's3a://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHH4954/ce17de38-54f8-45bf-82ba-ea1aec55061e/output/try-1/WHH4954.hard-filtered.gvcf.gz' to end in .vcf[.bgz, .gz]
2025-08-16 22:53:27.991 Hail: INFO: scanning VCF for sortedness...
2025-08-16 22:54:39.848 Hail: INFO: Coerced sorted VCF - no additional import work to do
2025-08-16 22:54:43.470 Hail: WARN: generated combiner save path of s3://precise-scratch/goypav/SG10K_Health/VDS/checkpoints/combiner-plans/vds-combiner-plan_c0b7181942e040568afae6658df3616303f612f32223e8df7a8757ad1707765f_0.2.134.json
2025-08-16 22:54:43.513 Hail: INFO: Running VDS combiner:
    VDS arguments: 0 datasets with 0 samples
    GVCF arguments: 1000 inputs/samples
    Branch factor: 

In [9]:
# Step 7: gVCF combiner batch 7
###

# Change the number of tasks hail is launching in parallel
hl._set_flags(spark_max_stage_parallelism='1000')

# Combine next batch of gVCFs
combiner = hl.vds.new_combiner(
    output_path=f'{vds_prefix}/SG10K_Health_gvcf1000-bf50-tr500k-sp1k_batch7.vds',
    temp_path=f'{vds_prefix}/checkpoints/',
    gvcf_paths=ls_gvcf[6000:7000],
    use_genome_default_intervals=True,
    reference_genome='GRCh38',
    branch_factor=50,
    target_records=500_000
)

combiner.run()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

2025-08-17 11:15:58.973 Hail: WARN: expected input file 's3a://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB5096/68eacbd6-7b21-47c8-9d7a-be588ac055d5/output/try-1/WHB5096.hard-filtered.gvcf.gz' to end in .vcf[.bgz, .gz]
2025-08-17 11:15:59.156 Hail: WARN: expected input file 's3a://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB5096/68eacbd6-7b21-47c8-9d7a-be588ac055d5/output/try-1/WHB5096.hard-filtered.gvcf.gz' to end in .vcf[.bgz, .gz]
2025-08-17 11:15:59.282 Hail: INFO: scanning VCF for sortedness...
2025-08-17 11:17:09.683 Hail: INFO: Coerced sorted VCF - no additional import work to do
2025-08-17 11:17:13.563 Hail: WARN: generated combiner save path of s3://precise-scratch/goypav/SG10K_Health/VDS/checkpoints/combiner-plans/vds-combiner-plan_c3a55af8c1f8c03ef75c2ebcecbc944c042ddfadf054ea2e3529179db6a57510_0.2.134.json
2025-08-17 11:17:13.596 Hail: INFO: Running VDS combiner:
    VDS arguments: 0 datasets with 0 samples
    GVCF arguments: 1000 inputs/samples
    Branch factor: 

In [10]:
# Step 8: gVCF combiner batch 8
###

# Change the number of tasks hail is launching in parallel
hl._set_flags(spark_max_stage_parallelism='1000')

# Combine next batch of gVCFs
combiner = hl.vds.new_combiner(
    output_path=f'{vds_prefix}/SG10K_Health_gvcf1000-bf50-tr500k-sp1k_batch8.vds',
    temp_path=f'{vds_prefix}/checkpoints/',
    gvcf_paths=ls_gvcf[7000:8000],
    use_genome_default_intervals=True,
    reference_genome='GRCh38',
    branch_factor=50,
    target_records=500_000
)

combiner.run()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

2025-08-17 23:13:06.240 Hail: WARN: expected input file 's3a://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB3963/bf796253-8f4c-43fb-b1bc-0589edaca791/output/try-1/WHB3963.hard-filtered.gvcf.gz' to end in .vcf[.bgz, .gz]
2025-08-17 23:13:06.421 Hail: WARN: expected input file 's3a://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB3963/bf796253-8f4c-43fb-b1bc-0589edaca791/output/try-1/WHB3963.hard-filtered.gvcf.gz' to end in .vcf[.bgz, .gz]
2025-08-17 23:13:06.541 Hail: INFO: scanning VCF for sortedness...
2025-08-17 23:14:17.537 Hail: INFO: Coerced sorted VCF - no additional import work to do
2025-08-17 23:14:21.378 Hail: WARN: generated combiner save path of s3://precise-scratch/goypav/SG10K_Health/VDS/checkpoints/combiner-plans/vds-combiner-plan_e548e3650b8131b50ba218caea2a01f5820501170376c4290865ca662dcc83db_0.2.134.json
2025-08-17 23:14:21.418 Hail: INFO: Running VDS combiner:
    VDS arguments: 0 datasets with 0 samples
    GVCF arguments: 1000 inputs/samples
    Branch factor: 

# Combine VDS

In [ ]:
# Use safer task parallelism (don't overload Spark)
hl._set_flags(spark_max_stage_parallelism='1000')  # reduce from 20000

# List of input VDS
ls_vds = [
    "s3a://precise-scratch/goypav/SG10K_Health/VDS/SG10K_Health_gvcf1000-bf50-tr500k-sp1k_batch5.vds",
    "s3a://precise-scratch/goypav/SG10K_Health/VDS/SG10K_Health_gvcf1000-bf50-tr500k-sp1k_batch6.vds",
    "s3a://precise-scratch/goypav/SG10K_Health/VDS/SG10K_Health_gvcf1000-bf50-tr500k-sp1k_batch7.vds",
    "s3a://precise-scratch/goypav/SG10K_Health/VDS/SG10K_Health_gvcf1000-bf50-tr500k-sp1k_batch8.vds",
]

# Define output and temp paths (using s3:// to ensure Hail uses async boto I/O)
# vds_prefix = 's3://precise-scratch/goypav/1KG/VDS'

combiner = hl.vds.new_combiner(
    output_path=f'{vds_prefix}/SG10K_Health_combined_batch5_6_7_8.bf2-tr500k-sp1k.n4000.vds',
    temp_path=f'{vds_prefix}/checkpoints/',
    vds_paths=ls_vds,
    use_genome_default_intervals=True,
    reference_genome='GRCh38',
    branch_factor=2,           # reduce concurrency → deeper tree
    target_records=500_000      # reduce total number of partitions
)

# Run the combination job
combiner.run()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

# Check VDS

In [13]:
# source
vds_prefix = 's3://precise-scratch/goypav/SG10K_Health/VDS/'

# input
vds_uri = vds_prefix + 'SG10K_Health_combined_batch5_6_7_8.bf2-tr500k-sp1k.n4000.vds'

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [14]:
# read VDS
vds = hl.vds.read_vds(vds_uri)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [15]:
# check reference_data
vds.reference_data.describe()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

----------------------------------------
Global fields:
    'ref_block_max_length': int32
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
----------------------------------------
Entry fields:
    'LEN': int32
    'DP': int32
    'GQ': int32
    'ICNT': array<int32>
    'MIN_DP': int32
    'SPL': array<int32>
    'LGT': call
    'LAD': array<int32>
    'END': int32
----------------------------------------
Column key: ['s']
Row key: ['locus']
----------------------------------------

In [16]:
# check variant_data
vds.variant_data.describe()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
----------------------------------------
Entry fields:
    'LA': array<int32>
    'LGT': call
    'LAD': array<int32>
    'LPL': array<int32>
    'RGQ': int32
    'gvcf_info': struct {
        DB: bool, 
        FS: float64, 
        FractionInformativeReads: float64, 
        LOD: float64, 
        MQ: float64, 
        MQRankSum: float64, 
        QD: float64, 
        R2_5P_bias: float64, 
        ReadPosRankSum: float64, 
        SOR: float64
    }
    'AF': array<float64>
    'DP': int32
    'F1R2': array<int32>
    'F2R1': array<int32>
    'GP': array<float64>
    'GQ': int32
    'ICNT': array<int32>
    'MB': array<int32>
    'MIN_DP': int32
    'PRI': array<float64>
    'PS': int32
    'SB': array<int32>
    'SPL': array<int32

In [17]:
# Count rows/cols in the variant_data MT
print(f"Reference genome: {vds.variant_data.locus.dtype.reference_genome.name}")
print(f"Number of samples: {vds.n_samples()}")
print(f"Number of variant partitions: {vds.variant_data.n_partitions()}")
print(f"Total number of variants: {vds.variant_data.count_rows():,}")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Reference genome: GRCh38
Number of samples: 4000
Number of variant partitions: 5781
Total number of variants: 294,446,810

In [18]:
hl.eval(vds.reference_data.ref_block_max_length)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

63690